# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aryan-0018/FlyRank-ML-W1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis + time window

One row represents one **daily observation for one content item for one client**, identified by `report_date`, `client_hash_id`, and `content_hash_id`. For this development work, I will use the **March 2026 mid-panel slice** (`month = '2026-03'`). The lane remains Refresh / Content Opportunity Scoring, so the page-level observation is the basis for building a priority score and supporting a content-review decision. The March slice is used for development rather than the final June 2026 month, which is treated as a sealed outcome window.

In [20]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [21]:
import duckdb

con = duckdb.connect()

print("DuckDB connected successfully.")

DuckDB connected successfully.


In [22]:
import os

os.environ["HF_TOKEN"] = HF_TOKEN

print("Warehouse access token configured for this session.")

Warehouse access token configured for this session.


In [23]:
!pip -q install duckdb huggingface_hub

In [24]:
con.execute("""
CREATE SECRET (
    TYPE huggingface,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face access configured in DuckDB.")

Hugging Face access configured in DuckDB.


In [25]:
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
TABLE_PATH = f"{WAREHOUSE}/fact_content_daily_performance/**/*.parquet"

print("Warehouse path configured.")

Warehouse path configured.


In [26]:
schema = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{TABLE_PATH}', hive_partitioning=true)
WHERE month = '2026-03'
LIMIT 1
""")

display(schema.df())

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields for my lane

**Features**
- `gsc_impressions` — knowable at the decision moment because it records observed Search Console impressions for the content item.
- `gsc_clicks` — knowable at the decision moment because it records observed Search Console clicks for the content item.
- `gsc_avg_position` — knowable at the decision moment because it records the observed Search Console average position for the content item.
- `ga4_pageviews` — knowable at the decision moment when `ga4_data_available IS TRUE`; it records observed page views for the content item.
- `ga4_engaged_sessions` — knowable at the decision moment when `ga4_data_available IS TRUE`; it records observed engaged sessions for the content item.

**Label / proxy**

- `decline_label` — an observed future outcome indicating whether a content item's total GSC impressions in the later outcome window (16–31 March 2026) are lower than its total GSC impressions in the decision window (1–15 March 2026). It is used only as the outcome, never as a predictive feature.

**Context**
- `report_date` — establishes the observation date and temporal ordering.
- `client_hash_id` — identifies the client grouping for validation and analysis, but is not a predictive feature.
- `content_hash_id` — identifies the content item and preserves the page-level grain, but is not a predictive feature.
- `gsc_data_available` — indicates whether GSC data is available for the observation.
- `ga4_data_available` — indicates whether GA4 data is available for the observation.
- `month` — partition and time-window context used to select the development slice.

**Excluded**
- `client_hash_id` and `content_hash_id` are excluded from the feature set because they are identifiers rather than predictive signals.
- Any field derived from, or only knowable because of, the outcome being predicted is excluded to prevent leakage.
- Product decision flags or previously calculated decision scores are excluded because they encode an existing decision rather than independent evidence.
- `decline_label` is excluded from the predictive feature set because it is the observed outcome itself and is unavailable at the decision moment; the deliberate leakage experiment demonstrates why including it produces an invalid result.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions",
]

print("Feature count:", len(feature_fields))
print("Features:", feature_fields)

Feature count: 5
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


### Five-feature frame

The initial feature frame uses five observed performance signals from the March 2026 development slice. These features are selected because they are available before a content-review decision and provide complementary measures of search exposure, search response, search position, and engagement.

**Available when?**
- `gsc_impressions` — available when `gsc_data_available IS TRUE`.
- `gsc_clicks` — available when `gsc_data_available IS TRUE`.
- `gsc_avg_position` — available when `gsc_data_available IS TRUE`.
- `ga4_pageviews` — available when `ga4_data_available IS TRUE`.
- `ga4_engaged_sessions` — available when `ga4_data_available IS TRUE`.

In [29]:
feature_frame = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_engaged_sessions
FROM read_parquet('{TABLE_PATH}', hive_partitioning=true)
WHERE month = '2026-03'
LIMIT 10
""")

display(feature_frame.df())

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,<NA>,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,<NA>,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,<NA>,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,<NA>,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,<NA>,<NA>


**Feature-frame observation:** The March 2026 frame contains the five selected features at the daily content-item level. The displayed rows show that GA4 fields can be unavailable for individual observations, reinforcing the need to use `ga4_data_available IS TRUE` when interpreting GA4-based features rather than treating missing values as zero.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain

This query checks whether the March 2026 slice contains duplicate combinations of `report_date`, `client_hash_id`, and `content_hash_id`, which tests the stated daily content-item grain.

In [30]:
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS n
FROM read_parquet('{TABLE_PATH}', hive_partitioning=true)
WHERE month = '2026-03'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""")

display(grain_check.df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


**Measured result:** No duplicate combinations of `report_date`, `client_hash_id`, and `content_hash_id` were found in the March 2026 slice, supporting the stated daily content-item grain.

### Query 2 — March 2026 row count and date span

This query verifies the size and observed date range of the development slice used in the contract.

In [31]:
count_dates = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{TABLE_PATH}', hive_partitioning=true)
WHERE month = '2026-03'
""")

display(count_dates.df())

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


**Measured result:** The March 2026 development slice contains 9,841,378 rows, covering the full observed period from 1 March 2026 to 31 March 2026.

### Query 3 — Data availability

This query checks how many March 2026 observations have GSC and GA4 data explicitly marked as available. The availability flags are tested with `IS TRUE` rather than treating zero values as evidence of missing data.

In [32]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available_rows
FROM read_parquet('{TABLE_PATH}', hive_partitioning=true)
WHERE month = '2026-03'
""")

display(availability_check.df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


**Measured result:** Of the 9,841,378 March 2026 rows, 3,611,061 have GSC data marked available, 413,966 have GA4 data marked available, and 364,347 have both available. Availability is determined from the explicit Boolean flags using `IS TRUE`, rather than treating zero-valued metrics as missingness.

### Deliberate leakage experiment

To demonstrate leakage, I use March 1–15 as the decision-time window and March 16–31 as the subsequent observed outcome window. The target is whether a content item's total search impressions decline in the later window relative to the decision-time window. The later-window outcome is not available at the decision moment and therefore must not be used as a feature.

I first evaluate a simple honest feature set using only decision-time observations. I then deliberately add the future outcome itself as a feature. This should produce an artificially strong score because the model is being given information that directly defines the target. I then remove the leaked field and retain the honest result.

In [33]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score

leakage_data = con.sql(f"""
WITH decision_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS decision_impressions,
        SUM(gsc_clicks) AS decision_clicks,
        AVG(gsc_avg_position) AS decision_avg_position,
        SUM(ga4_pageviews) AS decision_pageviews,
        SUM(ga4_engaged_sessions) AS decision_engaged_sessions
    FROM read_parquet('{TABLE_PATH}', hive_partitioning=true)
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
outcome_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS outcome_impressions
    FROM read_parquet('{TABLE_PATH}', hive_partitioning=true)
    WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    d.client_hash_id,
    d.content_hash_id,
    d.decision_impressions,
    d.decision_clicks,
    d.decision_avg_position,
    d.decision_pageviews,
    d.decision_engaged_sessions,
    o.outcome_impressions,
    CASE
        WHEN o.outcome_impressions < d.decision_impressions
        THEN 1
        ELSE 0
    END AS decline_label
FROM decision_window d
INNER JOIN outcome_window o
    ON d.client_hash_id = o.client_hash_id
   AND d.content_hash_id = o.content_hash_id
WHERE d.decision_impressions > 0
""")

leakage_df = leakage_data.df()

print("Rows with both decision and outcome windows:", len(leakage_df))
print("Decline rate:", leakage_df["decline_label"].mean())
display(leakage_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with both decision and outcome windows: 141467
Decline rate: 0.39636098878183607


,client_hash_id,content_hash_id,decision_impressions,decision_clicks,decision_avg_position,decision_pageviews,decision_engaged_sessions,outcome_impressions,decline_label
0,client_3ffa76342f366962,content_2d0506d130bb500d,7.0,0.0,4.142857,0.0,0.0,10.0,0
1,client_3ffa76342f366962,content_14ae08023ef326e8,9.0,0.0,9.666667,0.0,0.0,12.0,0
2,client_3ffa76342f366962,content_40f52f996b733f8f,6.0,0.0,8.200000,0.0,0.0,3.0,1
3,client_3ffa76342f366962,content_c6f14edf8567c310,3.0,0.0,2.333333,0.0,0.0,3.0,0
4,client_3ffa76342f366962,content_e5d1d2f323b8178d,4.0,0.0,4.666667,0.0,0.0,3.0,1


### Leakage experiment — honest versus leaked features

I first evaluate the decision-time features without using any future outcome information. I then deliberately add the observed future outcome label itself as a leaked feature. Because the leaked feature directly contains the information being predicted, its score should become artificially strong. The leaked feature is then removed, and the honest decision-time result is retained.

The purpose of this experiment is diagnostic: it demonstrates why a feature that is derived from the outcome cannot be used in a legitimate predictive workflow.

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

honest_features = [
    "decision_impressions",
    "decision_clicks",
    "decision_avg_position",
    "decision_pageviews",
    "decision_engaged_sessions",
]

X = leakage_df[honest_features]
y = leakage_df["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(max_iter=1000)
)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict_proba(X_test)[:, 1]
honest_auc = roc_auc_score(y_test, honest_predictions)

print("Honest ROC-AUC:", round(honest_auc, 4))

Honest ROC-AUC: 0.5869


### Deliberate leakage

For the leakage demonstration only, I add `decline_label` itself as a feature. This is intentionally invalid because the feature is the outcome being predicted. A legitimate model would never have access to this value at decision time.

In [35]:
leaked_features = honest_features + ["decline_label"]

X_leaked = leakage_df[leaked_features]

X_train_leaked, X_test_leaked, y_train_leaked, y_test_leaked = train_test_split(
    X_leaked,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaked_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(max_iter=1000)
)

leaked_model.fit(X_train_leaked, y_train_leaked)

leaked_predictions = leaked_model.predict_proba(X_test_leaked)[:, 1]
leaked_auc = roc_auc_score(y_test_leaked, leaked_predictions)

print("Honest ROC-AUC:", round(honest_auc, 4))
print("Leaked ROC-AUC:", round(leaked_auc, 4))

Honest ROC-AUC: 0.5869
Leaked ROC-AUC: 1.0


### Leakage removed

The leaked `decline_label` feature is removed. It is not part of the legitimate feature set because it is the outcome itself and is unavailable at the decision moment. The honest ROC-AUC is therefore the result retained for any future modelling work.

In [36]:
final_feature_set = honest_features

print("Final honest feature set:")
for feature in final_feature_set:
    print("-", feature)

print("Honest ROC-AUC retained:", round(honest_auc, 4))

Final honest feature set:
- decision_impressions
- decision_clicks
- decision_avg_position
- decision_pageviews
- decision_engaged_sessions
Honest ROC-AUC retained: 0.5869


**Leakage result:** The honest decision-time feature set achieved a ROC-AUC of 0.5869 on this diagnostic split. When `decline_label` itself was deliberately added as a feature, ROC-AUC increased to 1.0000. This confirms the leakage mechanism: the leaked feature directly contains the outcome information and is therefore invalid for prediction. After removing it, the honest ROC-AUC of 0.5869 is retained as the valid diagnostic result. This result is directional and should not be treated as evidence of production performance.

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This data supports measured, directional decision-support, but it cannot establish that a content change caused a later performance change.

First, client history is unbalanced, so the March 2026 slice does not necessarily represent the same depth or coverage of history for every client. This limits direct comparisons across clients and means later modelling should account for client-level history and use appropriate grouped validation.

Second, data availability is uneven. In the March 2026 slice, 3,611,061 of 9,841,378 rows have GSC data marked available, while only 413,966 have GA4 data available and 364,347 have both. Therefore, GA4-based features cannot be treated as universally available, and missingness must not be interpreted simply as zero performance.

Third, the warehouse contains overlapping time windows and the final month is reserved as a sealed outcome window. I will therefore avoid using future information when constructing features and will develop label logic on a mid-panel period rather than on the final June 2026 month.

Finally, the data does not by itself reveal editorial quality, search intent, business priorities, or whether an editor actually implemented a recommendation. A resulting score should therefore be treated as decision support rather than as proof that a page should be changed.

In [38]:
march_rows = 9841378
gsc_available_rows = 3611061
ga4_available_rows = 413966
both_available_rows = 364347

print("March rows:", march_rows)
print("GSC available:", gsc_available_rows)
print("GA4 available:", ga4_available_rows)
print("Both available:", both_available_rows)

March rows: 9841378
GSC available: 3611061
GA4 available: 413966
Both available: 364347


In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.